# FIFA World Cup 2026 Winner Predictor
## A Complete Machine Learning Pipeline Walkthrough

**Author:** Pablo Pena  
**Project:** [World Cup Winner Predictor](../README.md)

---

This notebook walks through the **entire data science pipeline** behind the World Cup prediction system:

| Step | Topic |
|------|-------|
| 1 | Problem framing & approach |
| 2 | Data loading & exploratory analysis |
| 3 | Feature engineering (point-in-time Elo) |
| 4 | Model 1 — Elo rating system |
| 5 | Model 2 — Dixon-Coles Poisson regression |
| 6 | Model 3 — Gradient boosting classifier |
| 7 | Ensemble combination |
| 8 | Out-of-time model validation |
| 9 | Monte Carlo tournament simulation |
| 10 | Results, interpretation & limitations |

> **Note on data:** Match results are real historical scores, manually curated into a compact 275-match dataset for reproducibility. See [data/README.md](../data/README.md) for details.

## 0. Setup

We import the project package (`wcp`) and configure plotting. The notebook assumes you run it from the `notebooks/` directory.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from wcp.config import ENSEMBLE_WEIGHTS, WORLD_CUP_2026_GROUPS
from wcp.data import load_matches
from wcp.evaluate import dataset_summary, evaluate_models
from wcp.features import FEATURE_COLS, build_training_frame, match_features, match_outcome
from wcp.models.elo import EloRatings
from wcp.models.dixon_coles import DixonColesModel
from wcp.models.ensemble import EnsemblePredictor
from wcp.priors import apply_elo_priors
from wcp.simulation import simulate_tournament
from wcp.train import train

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 100})
%matplotlib inline

print(f"Project root: {ROOT}")

---
## 1. Problem Framing

### The question

> *What is the probability that each nation wins the 2026 FIFA World Cup?*

This is **not** a single-match prediction problem. We need to:

1. **Model match outcomes** — estimate P(home win), P(draw), P(away win) for any two teams
2. **Simulate the tournament** — play out groups + knockout bracket thousands of times
3. **Aggregate** — count how often each team wins the final

### Why an ensemble?

No single model captures everything:

| Model | Strength | Weakness |
|-------|----------|----------|
| **Elo** | Simple, interpretable team rankings | Ignores goal margin, no score simulation |
| **Dixon-Coles** | Models actual goals scored/conceded | Slow to fit, needs many matches per team |
| **Gradient Boosting** | Captures non-linear patterns | Needs engineered features, less interpretable |

Combining them gives us ranking intuition (Elo), goal-based simulation (Dixon-Coles), and pattern recognition (GBM).

### 2026 World Cup format

- **48 teams** in 12 groups of 4
- Top 2 per group + 8 best third-place teams → **Round of 32**
- 5 knockout rounds → Final (8 matches to win)
- Hosts: USA, Mexico, Canada

---
## 2. Data Loading & Exploratory Analysis

Our dataset (`data/matches.csv`) contains **275 international matches** spanning:
- FIFA World Cups (1990–2022) — knockout rounds and key group games
- UEFA Euro 2024 — full knockout stage
- UEFA Nations League 2025 & recent friendlies (2023–2025)

Each row records: date, home team, away team, goals, tournament, and whether the venue was neutral.

In [ ]:
matches = load_matches()
summary = dataset_summary(matches)

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
for key, val in summary.items():
    print(f"  {key:25s} {val}")
print("=" * 50)

matches.head(10)

In [ ]:
# Matches by tournament
tournament_counts = matches["tournament"].value_counts()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

tournament_counts.plot(kind="bar", ax=axes[0], color="#1a535c", edgecolor="white")
axes[0].set_title("Matches by Tournament")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=30)

total_goals = matches["home_goals"] + matches["away_goals"]
sns.histplot(total_goals, bins=range(0, 13), ax=axes[1], color="#4ecdc4", edgecolor="white")
axes[1].set_title(f"Goals per Match (mean = {total_goals.mean():.2f})")
axes[1].set_xlabel("Total Goals")

outcomes = matches.apply(lambda r: match_outcome(r["home_goals"], r["away_goals"]), axis=1)
outcome_labels = {0: "Home Win", 1: "Draw", 2: "Away Win"}
outcome_counts = outcomes.map(outcome_labels).value_counts()
axes[2].pie(outcome_counts, labels=outcome_counts.index, autopct="%1.1f%%",
            colors=["#1a535c", "#ffe66d", "#ff6b6b"], startangle=90)
axes[2].set_title(f"Match Outcomes (draw rate = {summary['draw_rate']:.1%})")

plt.tight_layout()
plt.show()

In [ ]:
# World Cup scoring trends over time
wc = matches[matches["tournament"] == "World Cup"].copy()
wc["year"] = wc["date"].dt.year
wc["total_goals"] = wc["home_goals"] + wc["away_goals"]

yearly = wc.groupby("year").agg(
    matches=("total_goals", "count"),
    avg_goals=("total_goals", "mean"),
    draw_rate=("home_goals", lambda x: (x == wc.loc[x.index, "away_goals"]).mean()),
).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.bar(yearly["year"], yearly["matches"], color="#1a535c", alpha=0.4, label="Matches in dataset")
ax1.set_ylabel("Match count")
ax1.set_xlabel("World Cup Year")

ax2 = ax1.twinx()
ax2.plot(yearly["year"], yearly["avg_goals"], "o-", color="#ff6b6b", linewidth=2, label="Avg goals")
ax2.set_ylabel("Average goals per match")
ax2.set_ylim(0, 5)

ax1.set_title("World Cup Matches in Our Dataset — Scoring Trends")
fig.legend(loc="upper right", bbox_to_anchor=(0.88, 0.88))
plt.tight_layout()
plt.show()

yearly

### Key EDA takeaways

- International football is **low-scoring**: ~2.6 goals/match on average
- **Draws happen ~24%** of the time — any model must predict three outcomes, not just win/loss
- World Cup knockout matches in our data tend to be tighter (fewer goals) than group-stage friendlies
- We must be careful about **data leakage**: features must only use information available *before* each match

---
## 3. Feature Engineering

The gradient boosting model needs numeric features. We engineer them from **point-in-time Elo ratings** — meaning each match's features only reflect games that happened *before* it.

### Features used

| Feature | Description |
|---------|-------------|
| `home_elo` | Home team's Elo rating before the match |
| `away_elo` | Away team's Elo rating before the match |
| `elo_diff` | `home_elo - away_elo` (+ home advantage if not neutral) |
| `neutral` | 1 if neutral venue (World Cup), 0 if home advantage applies |
| `is_world_cup` | 1 if the match is a World Cup game |

### Target variable

- `0` = Home win
- `1` = Draw
- `2` = Away win

In [ ]:
elo_tracker = EloRatings()
train_df = build_training_frame(matches, elo_tracker)

print(f"Training frame shape: {train_df.shape}")
print(f"Features: {FEATURE_COLS}")
print(f"\nOutcome distribution:")
print(train_df["outcome"].map({0: "Home Win", 1: "Draw", 2: "Away Win"}).value_counts())

train_df[["home_team", "away_team", "home_elo", "away_elo", "elo_diff", "outcome"]].tail(8)

Notice how Elo ratings **evolve chronologically** — France's Elo climbs after Euro 2024 wins, Argentina's rises after the 2022 World Cup. This is the correct way to build features and avoids look-ahead bias.

---
## 4. Model 1 — Elo Rating System

Elo is a **dynamic rating system** originally designed for chess. After each match, ratings adjust based on whether the result was surprising.

### Update rule

$$E_{\text{new}} = E_{\text{old}} + K \cdot (S - E_{\text{expected}})$$

Where:
- $K = 40$ (learning rate — how much one match matters)
- $S \in \{0, 0.5, 1\}$ (actual result: loss, draw, win)
- $E_{\text{expected}} = \frac{1}{1 + 10^{-\Delta/400}}$ (expected score from rating difference)

### Home advantage

For non-neutral venues, the home team gets **+65 Elo points** before computing expected score. World Cup matches are neutral (`neutral=1`), so no bonus applies.

### Draw modeling

Elo alone only gives win probability. We add an empirical draw model:

$$P(\text{draw}) = 0.26 \times e^{-|\Delta| / 600}$$

Closely matched teams draw more often; mismatches rarely do.

In [ ]:
elo = EloRatings()
elo.fit(matches)

top_elo = sorted(elo.ratings.items(), key=lambda x: -x[1])[:15]
elo_df = pd.DataFrame(top_elo, columns=["team", "elo"])

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(elo_df["team"][::-1], elo_df["elo"][::-1], color="#1a535c", edgecolor="white")
ax.axvline(1500, color="red", linestyle="--", alpha=0.5, label="Baseline (1500)")
ax.set_xlabel("Elo Rating")
ax.set_title("Top 15 Teams by Elo Rating (trained on full dataset)")
ax.legend()
plt.tight_layout()
plt.show()

elo_df

In [ ]:
# Example: what does Elo predict for France vs Germany?
home, away = "France", "Germany"
p_h, p_d, p_a = elo.win_prob(home, away, neutral=True)

print(f"{home} (Elo {elo.get(home):.0f}) vs {away} (Elo {elo.get(away):.0f})")
print(f"  P({home} win) = {p_h:.1%}")
print(f"  P(Draw)      = {p_d:.1%}")
print(f"  P({away} win) = {p_a:.1%}")

# Visualize how draw probability changes with rating gap
diffs = np.arange(-600, 601, 10)
draw_probs = [0.26 * np.exp(-abs(d) / 600) for d in diffs]
win_probs = [1 / (1 + 10 ** (-d / 400)) for d in diffs]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(diffs, win_probs, label="P(stronger team wins)", color="#1a535c")
ax.plot(diffs, draw_probs, label="P(draw)", color="#ffe66d")
ax.set_xlabel("Elo difference (home − away)")
ax.set_ylabel("Probability")
ax.set_title("Elo Win & Draw Probabilities vs Rating Gap")
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Model 2 — Dixon-Coles Poisson Regression

The Dixon-Coles model (1997) treats goals as **Poisson-distributed** with team-specific attack and defense parameters.

### Expected goals

$$\lambda_{\text{home}} = \exp(\alpha_{\text{home}} - \delta_{\text{away}} + \gamma)$$
$$\lambda_{\text{away}} = \exp(\alpha_{\text{away}} - \delta_{\text{home}})$$

Where:
- $\alpha_i$ = attack strength of team $i$ (higher = scores more)
- $\delta_i$ = defense strength of team $i$ (higher = concedes more)
- $\gamma$ = home advantage parameter (0 for neutral venues)

### Low-score correction

Standard Poisson **over-predicts** 0-0 and 1-1 draws. Dixon & Coles add a correction factor $\tau$ for scores ≤ 1:

| Score | τ factor |
|-------|----------|
| 0-0 | $1 - \lambda_h \lambda_a \rho$ |
| 1-1 | $1 - \rho$ |

With $\rho \approx -0.13$ (negative correlation between low scores).

### Time decay

Recent matches matter more: $w(t) = e^{-\xi(t_{\max} - t)}$ with $\xi = 0.0018$.

Parameters are estimated via **maximum likelihood** (L-BFGS-B optimizer).

In [ ]:
print("Fitting Dixon-Coles model (this takes ~10 seconds)...")
dc = DixonColesModel()
dc.fit(matches)
print(f"Home advantage (γ): {dc.home_adv:.3f}")
print(f"Low-score correlation (ρ): {dc.rho:.3f}")
print(f"Teams with parameters: {len(dc.attack)}")

In [ ]:
# Attack vs Defense scatter plot
teams = sorted(dc.attack.keys())
att = [dc.attack[t] for t in teams]
deff = [dc.defense[t] for t in teams]

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(att, deff, alpha=0.5, s=40, c="#1a535c")

# Label top teams
for t in ["France", "Spain", "Germany", "Brazil", "Argentina", "England"]:
    if t in dc.attack:
        ax.annotate(t, (dc.attack[t], dc.defense[t]), fontsize=9, fontweight="bold")

ax.axhline(0, color="gray", linestyle="--", alpha=0.3)
ax.axvline(0, color="gray", linestyle="--", alpha=0.3)
ax.set_xlabel("Attack strength (α)")
ax.set_ylabel("Defense strength (δ) — lower is better")
ax.set_title("Dixon-Coles: Attack vs Defense Parameters")
plt.tight_layout()
plt.show()

In [ ]:
# Score probability matrix for France vs Germany
home, away = "France", "Germany"
score_mat = dc.score_matrix(home, away, neutral=True)
lam_h, lam_a = dc.expected_goals(home, away, neutral=True)
p_h, p_d, p_a = dc.win_prob(home, away, neutral=True)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(score_mat, cmap="YlOrRd", origin="lower")
ax.set_xlabel(f"{away} goals")
ax.set_ylabel(f"{home} goals")
ax.set_title(f"{home} vs {away}\nλ_home={lam_h:.2f}, λ_away={lam_a:.2f}")
plt.colorbar(im, label="P(score)")

for i in range(score_mat.shape[0]):
    for j in range(score_mat.shape[1]):
        if score_mat[i, j] > 0.03:
            ax.text(j, i, f"{score_mat[i,j]:.2f}", ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.show()

print(f"P({home} win) = {p_h:.1%}  |  P(Draw) = {p_d:.1%}  |  P({away} win) = {p_a:.1%}")

The score matrix is what makes Dixon-Coles essential for **tournament simulation** — we can sample realistic scores (e.g., 2-1, 0-0) rather than just picking a winner.

---
## 6. Model 3 — Gradient Boosting Classifier

We use scikit-learn's `HistGradientBoostingClassifier` to learn **non-linear relationships** between Elo-based features and match outcomes.

### Why gradient boosting?

- Handles interactions (e.g., "high Elo diff + World Cup = even more likely home win")
- Robust with small datasets (275 matches)
- Outputs calibrated probabilities via isotonic regression

### Training setup

- **300 boosting iterations**, max depth 5
- **Time-series cross-validation** (3 folds) — never trains on future data
- **Isotonic calibration** to ensure predicted probabilities are well-calibrated

In [ ]:
from wcp.models.lightgbm_model import LightGBMPredictor

gbm = LightGBMPredictor()
gbm.fit(train_df, train_df["outcome"])

# Feature importance via a simple correlation analysis
corr = train_df[FEATURE_COLS + ["outcome"]].corr()["outcome"].drop("outcome")

fig, ax = plt.subplots(figsize=(8, 3))
corr.plot(kind="barh", ax=ax, color="#4ecdc4", edgecolor="white")
ax.set_title("Feature Correlation with Match Outcome")
ax.set_xlabel("Pearson correlation")
ax.axvline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

print("Correlation with outcome:")
print(corr.sort_values(ascending=False).to_string())

In [ ]:
# Compare all three models on a single matchup
home, away = "Spain", "England"
feats = match_features(home, away, elo, neutral=True, is_wc=True)

models = {
    "Elo": elo.win_prob(home, away, neutral=True),
    "Dixon-Coles": dc.win_prob(home, away, neutral=True),
    "Gradient Boosting": gbm.predict_proba(feats),
}

comparison = pd.DataFrame(models, index=["Home Win", "Draw", "Away Win"]).T
comparison.columns = [f"P({home})", "P(Draw)", f"P({away})"]

print(f"\n{home} vs {away} — Model Comparison\n")
display(comparison.style.format("{:.1%}").background_gradient(cmap="YlOrRd", axis=None))

comparison.plot(kind="bar", figsize=(10, 4), rot=0, edgecolor="white")
plt.title(f"{home} vs {away}: Predicted Outcome Probabilities by Model")
plt.ylabel("Probability")
plt.legend(bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

---
## 7. Ensemble Combination

We combine the three models via a **weighted average** of their probability vectors:

$$P_{\text{ensemble}} = w_1 P_{\text{Elo}} + w_2 P_{\text{DC}} + w_3 P_{\text{GBM}}$$

Weights were chosen based on out-of-time validation performance:

In [ ]:
print("Ensemble weights:")
for model, weight in ENSEMBLE_WEIGHTS.items():
    print(f"  {model:20s} {weight:.0%}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.pie(ENSEMBLE_WEIGHTS.values(), labels=ENSEMBLE_WEIGHTS.keys(), autopct="%1.0f%%",
       colors=["#1a535c", "#4ecdc4", "#ff6b6b"], startangle=90)
ax.set_title("Ensemble Model Weights")
plt.show()

In [ ]:
# Train the full ensemble and apply 2026 Elo priors
predictor = EnsemblePredictor()
predictor.fit(matches, train_df)
apply_elo_priors(predictor.elo)

# Show ensemble prediction for Spain vs England
p_h, p_d, p_a = predictor.match_prob(home, away, neutral=True)
print(f"\nEnsemble prediction: {home} vs {away}")
print(f"  P({home} win) = {p_h:.1%}")
print(f"  P(Draw)      = {p_d:.1%}")
print(f"  P({away} win) = {p_a:.1%}")

### Elo priors for 2026 teams

Many World Cup 2026 participants have few matches in our dataset. Without adjustment, they'd default to Elo = 1500 (average), which overstates weaker teams and understates strong ones.

We blend trained Elo with **FIFA ranking-derived priors** for all 48 qualified nations (see `wcp/priors.py`).

---
## 8. Model Validation — Out-of-Time Backtesting

The gold standard for sports prediction: **train on the past, test on the future**.

- **Training set:** All matches before 2018
- **Test set:** Matches from 2018 onward (WC 2018, WC 2022, Euro 2024, recent internationals)

### Metrics

- **Log loss** (primary): $-\frac{1}{N}\sum \log P(y_{\text{true}})$. Lower is better. Penalizes confident wrong predictions heavily.
- **Accuracy**: Fraction of correct outcome predictions.

In [ ]:
eval_df = evaluate_models(matches, test_from_year=2018)
eval_df = eval_df.sort_values("log_loss")

print(f"Test set: {eval_df['n_matches'].iloc[0]} matches (2018–2025)\n")
display(eval_df.style.format({"accuracy": "{:.1%}", "log_loss": "{:.3f}"})
              .background_gradient(subset=["log_loss"], cmap="RdYlGn_r"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ["#ff6b6b" if m == "Ensemble" else "#4ecdc4" for m in eval_df["model"]]

axes[0].barh(eval_df["model"], eval_df["accuracy"], color=colors, edgecolor="white")
axes[0].set_xlabel("Accuracy")
axes[0].set_title("Out-of-Time Accuracy (2018+)")
axes[0].set_xlim(0, 0.7)
for i, (_, row) in enumerate(eval_df.iterrows()):
    axes[0].text(row["accuracy"] + 0.01, i, f"{row['accuracy']:.1%}", va="center")

axes[1].barh(eval_df["model"], eval_df["log_loss"], color=colors, edgecolor="white")
axes[1].set_xlabel("Log Loss (lower = better)")
axes[1].set_title("Out-of-Time Log Loss (2018+)")
axes[1].invert_xaxis()
for i, (_, row) in enumerate(eval_df.iterrows()):
    axes[1].text(row["log_loss"] - 0.02, i, f"{row['log_loss']:.3f}", va="center", ha="right")

plt.suptitle("Model Backtesting Results", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### Validation insights

- **Elo** is a strong baseline — simple but hard to beat on log loss
- **Dixon-Coles** adds value for goal simulation even if its win/draw/loss classification is weaker
- **Gradient Boosting** overfits on small data (275 matches) — calibration helps but limited sample size hurts
- The **ensemble** balances these trade-offs, achieving the best or near-best accuracy

> With only 114 test matches, these metrics have high variance. A production system would use 10,000+ matches.

---
## 9. Monte Carlo Tournament Simulation

Now we simulate the **entire 2026 World Cup** thousands of times.

### Algorithm

For each simulation:

1. **Group stage** — play all 72 group matches (12 groups × 6 matches each)
   - Sample scores from Dixon-Coles Poisson model
   - Award 3/1/0 points, rank by points → goal difference → goals scored
   - Top 2 per group + 8 best third-place teams advance (32 teams)

2. **Knockout rounds** — Round of 32 → Round of 16 → QF → SF → Final
   - Teams seeded by Elo rating
   - Sample match scores; ties resolved by Elo-weighted penalty shootout

3. **Record the champion**

After $N$ simulations:

$$P(\text{team wins World Cup}) = \frac{\text{times team won}}{N}$$

In [ ]:
# Display the official 2026 draw
print("2026 FIFA World Cup — Group Draw\n")
for group, teams in sorted(WORLD_CUP_2026_GROUPS.items()):
    elos = [predictor.elo.get(t) for t in teams]
    avg = np.mean(elos)
    print(f"  Group {group} (avg Elo {avg:.0f}): {' · '.join(teams)}")

In [ ]:
# Group strength comparison
group_stats = []
for group, teams in WORLD_CUP_2026_GROUPS.items():
    elos = [predictor.elo.get(t) for t in teams]
    group_stats.append({"group": group, "avg_elo": np.mean(elos), "max_elo": max(elos), "min_elo": min(elos)})

gs = pd.DataFrame(group_stats).sort_values("avg_elo", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(gs["group"].apply(lambda g: f"Group {g}"), gs["avg_elo"], color="#1a535c", edgecolor="white")
ax.set_xlabel("Average Elo Rating")
ax.set_title("2026 World Cup — Group Difficulty")
plt.tight_layout()
plt.show()

gs.sort_values("avg_elo", ascending=False)

In [ ]:
N_SIMS = 10_000
print(f"Running {N_SIMS:,} tournament simulations...")

results = simulate_tournament(predictor, n_sims=N_SIMS, seed=42)

pred_df = pd.DataFrame([
    {"team": team, "win_probability": prob, "elo": predictor.elo.get(team)}
    for team, prob in results.items()
]).sort_values("win_probability", ascending=False)

pred_df["win_pct"] = (pred_df["win_probability"] * 100).round(2)
pred_df["rank"] = range(1, len(pred_df) + 1)

print(f"\nTop 15 contenders:\n")
display(pred_df[["rank", "team", "win_pct", "elo"]].head(15).style.format({"win_pct": "{:.2f}%", "elo": "{:.0f}"}))

In [ ]:
# Win probability chart
top15 = pred_df.head(15)
colors = ["#ff6b6b" if i == 0 else "#4ecdc4" if i < 3 else "#7f8c8d" for i in range(len(top15))]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top15["team"][::-1], top15["win_pct"][::-1], color=colors[::-1], edgecolor="white")
ax.set_xlabel("Win Probability (%)")
ax.set_title(f"2026 FIFA World Cup — Title Probabilities ({N_SIMS:,} simulations)", fontweight="bold")

for bar, pct in zip(bars, top15["win_pct"][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2, f"{pct:.1f}%", va="center")

plt.tight_layout()
plt.show()

In [ ]:
# Probability concentration: how "open" is this World Cup?
top5_share = pred_df.head(5)["win_probability"].sum()
top10_share = pred_df.head(10)["win_probability"].sum()

print(f"Top 5 teams account for {top5_share:.1%} of all probability mass")
print(f"Top 10 teams account for {top10_share:.1%} of all probability mass")
print(f"\nPredicted champion: {pred_df.iloc[0]['team']} ({pred_df.iloc[0]['win_pct']:.1f}%)")
print(f"Most likely final:  {pred_df.iloc[0]['team']} vs {pred_df.iloc[1]['team']}")

# Cumulative probability curve
pred_df["cumulative"] = pred_df["win_probability"].cumsum()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(pred_df) + 1), pred_df["cumulative"], "o-", color="#1a535c", markersize=3)
ax.axhline(0.5, color="red", linestyle="--", alpha=0.5, label="50% of probability mass")
ax.axhline(0.8, color="orange", linestyle="--", alpha=0.5, label="80% of probability mass")
ax.set_xlabel("Number of teams (ranked by win probability)")
ax.set_ylabel("Cumulative win probability")
ax.set_title("How Concentrated Are the Title Chances?")
ax.legend()
plt.tight_layout()
plt.show()

---
## 10. Results & Interpretation

### What the model tells us

1. **Germany and France** are clear favorites — together they account for ~40-45% of simulated titles
2. **Spain and Brazil** form a second tier of contenders (~7-8% each)
3. The tournament is relatively **open** compared to, say, club football — no team exceeds 25% win probability
4. **Host advantage** is modest in our model (small Elo boost for USA/Mexico/Canada)

### What the model cannot tell us

- **Injuries** to key players (e.g., if a star striker is ruled out)
- **Tactical changes** under new managers
- **Momentum** within a tournament (hot streaks, penalty shootout psychology)
- **Exact bracket paths** — our simplified knockout pairing differs from FIFA's official bracket

---
## 11. Limitations & Future Improvements

| Limitation | Impact | Fix |
|------------|--------|-----|
| Small dataset (275 matches) | High variance in model estimates | Ingest full Kaggle dataset (48k+ matches) |
| Manual data curation | Selection bias toward famous matches | Automated API/scraping pipeline |
| No player-level data | Misses injury/form effects | xG models with player rosters |
| Simplified bracket | Knockout paths may differ from reality | Implement exact FIFA bracket rules |
| Static weights | Ensemble weights not re-optimized | Bayesian optimization on validation set |
| No confederation effects | Missing travel/familiarity factors | Add confederation & climate features |

### Next steps for a production system

1. **Expand data** to all international matches since 1990 (~15,000 matches)
2. **Player-level xG** from club football as a leading indicator
3. **Bayesian hierarchical models** for better uncertainty quantification
4. **Live updating** during the tournament as results come in
5. **Proper bracket simulation** matching FIFA's exact Round of 32 draw rules

---
## 12. Save Results

Export predictions and figures for the portfolio.

In [ ]:
from wcp.config import FIGURES_DIR, RESULTS_DIR
from wcp import viz

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

pred_df.to_csv(RESULTS_DIR / "predictions_2026.csv", index=False)
eval_df.to_csv(RESULTS_DIR / "model_evaluation.csv", index=False)

viz.plot_win_probabilities(results, predictor, save_path=FIGURES_DIR / "win_probabilities.png")
viz.plot_elo_rankings(predictor, save_path=FIGURES_DIR / "elo_rankings.png")
viz.plot_attack_defense(predictor, save_path=FIGURES_DIR / "attack_defense.png")
viz.plot_group_strength(predictor, save_path=FIGURES_DIR / "group_strength.png")
viz.plot_model_comparison(eval_df, save_path=FIGURES_DIR / "model_comparison.png")
viz.plot_goals_distribution(matches, save_path=FIGURES_DIR / "goals_distribution.png")

print("Saved to outputs/:")
print(f"  {RESULTS_DIR / 'predictions_2026.csv'}")
print(f"  {RESULTS_DIR / 'model_evaluation.csv'}")
print(f"  {FIGURES_DIR}/ (6 figures)")

---

*End of notebook. For the full methodology, see [docs/METHODOLOGY.md](../docs/METHODOLOGY.md). To run predictions from the command line: `python main.py predict --sims 10000`*